# 03 InSAR Time-Series Validation Against GNSS

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roymustang11/InSAR-Benchmark-Lab/blob/main/notebooks/03_mintpy_timeseries_validation.ipynb)

This notebook defines the validation workflow for MintPy, OPERA, HyP3, or ARIA-derived displacement time series.

**Data mode:** `DEMONSTRATION_DATA = True`. Values in this notebook are controlled examples for method development, not Central Valley deformation measurements.


## Goal

For a real benchmark, this notebook will answer:

> How closely does an InSAR line-of-sight displacement time series match an independent GNSS displacement time series at validation locations?

The workflow aligns dates, computes residuals, reports validation metrics, and produces a figure that can be repeated for each station or pixel target.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/roymustang11/InSAR-Benchmark-Lab.git"
DEMONSTRATION_DATA = True

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path("/content/InSAR-Benchmark-Lab")
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project_root)])
else:
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Example-data mode: {DEMONSTRATION_DATA}")


In [ ]:
from insar_benchmark_lab.config import load_study_area_config

config = load_study_area_config(project_root / "configs" / "central_valley_subsidence.yml")
config


## Example Time Series

The table below matches the target validation schema: dates, InSAR displacement, GNSS displacement, and per-date InSAR uncertainty. Replace this section with product-derived values after OPERA/MintPy extraction is implemented.


In [ ]:
from datetime import date

import numpy as np
import pandas as pd

dates = pd.date_range("2020-01-01", periods=12, freq="90D")
years = np.arange(len(dates)) * 90 / 365.25
gnss_mm = -24.0 * years + 1.2 * np.sin(np.linspace(0, 2 * np.pi, len(dates)))
insar_mm = gnss_mm + np.array([0.4, -0.8, 1.1, -1.3, 0.7, -0.4, 0.6, -0.9, 1.2, -0.5, 0.3, -0.7])
sigma_mm = np.full(len(dates), 2.5)

validation_table = pd.DataFrame(
    {
        "date": dates.date,
        "insar_los_mm": insar_mm,
        "gnss_projected_los_mm": gnss_mm,
        "insar_sigma_mm": sigma_mm,
    }
)
validation_table.head()


## Validation Metrics

The metric set is intentionally small and interpretable: RMSE, MAE, correlation, mean bias, endpoint velocity difference, and uncertainty coverage.


In [ ]:
from insar_benchmark_lab.metrics import (
    correlation,
    mae,
    rmse,
    trend_bias,
    uncertainty_coverage,
    velocity_difference,
)

date_strings = [item.isoformat() for item in validation_table["date"]]
insar = validation_table["insar_los_mm"].to_numpy()
gnss = validation_table["gnss_projected_los_mm"].to_numpy()
sigma = validation_table["insar_sigma_mm"].to_numpy()

validation_metrics = {
    "rmse_mm": rmse(gnss, insar),
    "mae_mm": mae(gnss, insar),
    "correlation": correlation(gnss, insar),
    "mean_gnss_minus_insar_bias_mm": trend_bias(gnss, insar),
    "insar_minus_gnss_velocity_difference_mm_per_year": velocity_difference(date_strings, insar, gnss),
    "one_sigma_coverage_fraction": uncertainty_coverage(gnss, insar, sigma),
}

pd.Series(validation_metrics).to_frame("value")


In [ ]:
import matplotlib.pyplot as plt

residual = validation_table["insar_los_mm"] - validation_table["gnss_projected_los_mm"]
figure_panels, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, constrained_layout=True)

axes[0].plot(validation_table["date"], validation_table["insar_los_mm"], marker="o", label="InSAR LOS example")
axes[0].plot(validation_table["date"], validation_table["gnss_projected_los_mm"], marker="s", label="GNSS projected LOS example")
axes[0].fill_between(
    validation_table["date"],
    validation_table["insar_los_mm"] - validation_table["insar_sigma_mm"],
    validation_table["insar_los_mm"] + validation_table["insar_sigma_mm"],
    alpha=0.2,
    label="InSAR +/- 1 sigma",
)
axes[0].set_ylabel("LOS displacement (mm)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color="black", linewidth=1)
axes[1].bar(validation_table["date"], residual, width=45, color="#d55e00")
axes[1].set_ylabel("InSAR - GNSS residual (mm)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

figure_panels.suptitle("InSAR-vs-GNSS Validation Workflow")
plt.show()


## Transition To Real Data

To convert this notebook to a measured result:

1. extract OPERA/MintPy displacement values at GNSS station locations,
2. project GNSS east/north/up motion into the satellite line of sight,
3. align dates using the shared `timeseries.align_by_date` utility,
4. rerun this metric and plotting section without changing the interpretation criteria.
